You are working on an e-commerce platform (Amazon-like).

Every day:

Customers place orders

Some customers are new (first-time buyers)

Some customers are repeat buyers

🎯 Goal

For each day, find:

Number of new customers

Number of repeat customers

🧱 Table Structure

CREATE TABLE customer_orders (
    order_id INTEGER,
    customer_id INTEGER,
    order_date TEXT,      -- YYYY-MM-DD
    order_amount INTEGER
);

In [0]:
WITH first_order AS
(
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date
    FROM customer_orders
    GROUP BY customer_id
)

SELECT
    c.order_date,

    COUNT(DISTINCT CASE
            WHEN c.order_date = f.first_order_date
            THEN c.customer_id
        END) AS new_customers,

    COUNT(DISTINCT CASE
            WHEN c.order_date > f.first_order_date
            THEN c.customer_id
        END) AS repeat_customers

FROM customer_orders c
JOIN first_order f
ON c.customer_id = f.customer_id

GROUP BY c.order_date
ORDER BY c.order_date;

In [0]:
from pyspark.sql import functions as F

# Step 1: Find first purchase date of every customer
first_order = (
    df.groupBy("customer_id")
      .agg(F.min("order_date").alias("first_order_date"))
)

# Step 2: Join with original table
joined = df.join(first_order, "customer_id")

# Step 3: Aggregate daily counts
result = (
    joined.groupBy("order_date")
          .agg(
              F.countDistinct(
                  F.when(
                      F.col("order_date") == F.col("first_order_date"),
                      F.col("customer_id")
                  )
              ).alias("new_customers"),

              F.countDistinct(
                  F.when(
                      F.col("order_date") > F.col("first_order_date"),
                      F.col("customer_id")
                  )
              ).alias("repeat_customers")
          )
          .orderBy("order_date")
)

result.show()

In [0]:
WITH cte AS
(
    SELECT *,
           MIN(order_date) OVER(PARTITION BY customer_id) AS first_order_date
    FROM customer_orders
)

SELECT
    order_date,
    COUNT(DISTINCT CASE WHEN order_date = first_order_date THEN customer_id END) AS new_customers,
    COUNT(DISTINCT CASE WHEN order_date > first_order_date THEN customer_id END) AS repeat_customers
FROM cte
GROUP BY order_date
ORDER BY order_date;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w = Window.partitionBy("customer_id")

result = (
    df.withColumn("first_order_date", F.min("order_date").over(w))
      .groupBy("order_date")
      .agg(
          F.countDistinct(
              F.when(
                  F.col("order_date") == F.col("first_order_date"),
                  F.col("customer_id")
              )
          ).alias("new_customers"),

          F.countDistinct(
              F.when(
                  F.col("order_date") > F.col("first_order_date"),
                  F.col("customer_id")
              )
          ).alias("repeat_customers")
      )
      .orderBy("order_date")
)

result.show()